# 🎬 Netflix Customer Churn Analysis

**Author:** Alyana Bhanu Chandar  
**Dataset:** Netflix Customer Churn  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn  

---

## 📋 Project Overview

This notebook walks through a complete beginner-friendly data analysis of a Netflix Customer Churn dataset.  
**Churn** means a customer has cancelled (or stopped using) the service. Understanding why customers churn helps businesses retain them.

### 🗂️ Table of Contents
1. [Import Libraries](#1)
2. [Load the Dataset](#2)
3. [Data Overview & Basic Info](#3)
4. [Data Cleaning](#4)
5. [Exploratory Data Analysis (EDA)](#5)
   - 5.1 Churn Distribution
   - 5.2 Age Distribution
   - 5.3 Gender Analysis
   - 5.4 Subscription Type Analysis
   - 5.5 Region Analysis
   - 5.6 Device Usage
   - 5.7 Payment Method
   - 5.8 Favourite Genre
6. [Feature Analysis vs Churn](#6)
   - 6.1 Watch Hours vs Churn
   - 6.2 Last Login Days vs Churn
   - 6.3 Monthly Fee vs Churn
   - 6.4 Number of Profiles vs Churn
   - 6.5 Avg Watch Time per Day vs Churn
7. [Correlation Heatmap](#7)
8. [Churn Rate by Category](#8)
9. [Key Insights](#9)
10. [Conclusion](#10)

---
<a id='1'></a>
## 1. 📦 Import Libraries

In [ ]:
# Standard data manipulation libraries
import pandas as pd
import numpy as np

# Visualisation libraries
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Display settings
import warnings
warnings.filterwarnings('ignore')          # Suppress harmless warnings

%matplotlib inline                         # Render plots inside the notebook

# Set a clean, consistent visual theme for all plots
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print('✅ All libraries imported successfully!')

---
<a id='2'></a>
## 2. 📂 Load the Dataset

In [ ]:
# Load the CSV file into a pandas DataFrame
df = pd.read_csv('netflix_customer_churn.csv')

print(f'Dataset loaded successfully!')
print(f'Shape: {df.shape[0]} rows  ×  {df.shape[1]} columns')

In [ ]:
# Preview the first 5 rows to understand the data structure
df.head()

In [ ]:
# Preview the last 5 rows
df.tail()

---
<a id='3'></a>
## 3. 🔍 Data Overview & Basic Info

Before analysing, we always check:
- **Shape** – number of rows and columns
- **Data types** – numeric, string, etc.
- **Missing values** – gaps in data we need to handle
- **Basic statistics** – mean, min, max, std of numeric columns

In [ ]:
# Column names and data types
print('=== Column Info ===')
df.info()

In [ ]:
# Check for missing (null) values in every column
print('=== Missing Values per Column ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)

In [ ]:
# Check for duplicate rows
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Statistical summary of all NUMERIC columns
print('=== Descriptive Statistics (Numeric Columns) ===')
df.describe().round(2)

In [ ]:
# Statistical summary of CATEGORICAL columns
print('=== Descriptive Statistics (Categorical Columns) ===')
df.describe(include='object').T

In [ ]:
# Unique values for each categorical column
cat_cols = ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']

for col in cat_cols:
    print(f'{col}: {sorted(df[col].unique().tolist())}')

### 📌 Column Descriptions

| Column | Description |
|---|---|
| `customer_id` | Unique customer identifier (UUID) |
| `age` | Customer age (18 – 70) |
| `gender` | Gender: Male / Female / Other |
| `subscription_type` | Plan: Basic / Standard / Premium |
| `watch_hours` | Total hours watched |
| `last_login_days` | Days since last login (0 – 60) |
| `region` | Geographic region |
| `device` | Device used most: TV / Mobile / Laptop / Tablet / Desktop |
| `monthly_fee` | Monthly subscription fee (USD) |
| `churned` | **Target variable** – 1 = churned, 0 = active |
| `payment_method` | Payment method used |
| `number_of_profiles` | Profiles under the account (1 – 5) |
| `avg_watch_time_per_day` | Average hours watched per day |
| `favorite_genre` | Customer's most-watched genre |

---
<a id='4'></a>
## 4. 🧹 Data Cleaning

Even though this dataset has no missing values, good practice is to:
- Verify data ranges make sense (e.g., age shouldn't be negative)
- Remove unnecessary columns
- Create useful derived columns for analysis

In [ ]:
# Drop 'customer_id' — it's just an ID and has no analytical value
df.drop(columns=['customer_id'], inplace=True)
print('Dropped customer_id column.')
print(f'New shape: {df.shape}')

In [ ]:
# Validate numeric ranges
print('Age range:              ', df['age'].min(), '–', df['age'].max())
print('Watch hours range:      ', df['watch_hours'].min(), '–', df['watch_hours'].max())
print('Last login days range:  ', df['last_login_days'].min(), '–', df['last_login_days'].max())
print('Monthly fee range:      ', df['monthly_fee'].min(), '–', df['monthly_fee'].max())
print('Number of profiles:     ', df['number_of_profiles'].min(), '–', df['number_of_profiles'].max())
print('Avg watch time/day:     ', df['avg_watch_time_per_day'].min(), '–', df['avg_watch_time_per_day'].max())

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────────────

# 1. Age group buckets for cleaner visualisation
bins   = [17, 25, 35, 45, 55, 70]
labels = ['18-25', '26-35', '36-45', '46-55', '56-70']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)

# 2. Inactivity level based on last_login_days
#    ≤ 15 days → Active | 16–30 → Moderate | > 30 → Inactive
df['inactivity_level'] = pd.cut(
    df['last_login_days'],
    bins=[-1, 15, 30, 60],
    labels=['Active (≤15d)', 'Moderate (16-30d)', 'Inactive (>30d)']
)

# 3. Churned label (readable version of the 0/1 flag)
df['churn_label'] = df['churned'].map({1: 'Churned', 0: 'Active'})

print('Feature engineering complete. New columns added:')
print(df[['age_group', 'inactivity_level', 'churn_label']].head())

In [ ]:
# Final cleaned dataset snapshot
print(f'Final dataset shape: {df.shape}')
df.head(3)

---
<a id='5'></a>
## 5. 📊 Exploratory Data Analysis (EDA)

EDA helps us understand the distribution of each variable and spot patterns before doing deeper analysis.

### 5.1 🔴 Churn Distribution (Target Variable)

In [ ]:
# Count and percentage of churned vs active customers
churn_counts = df['churn_label'].value_counts()
churn_pct    = df['churn_label'].value_counts(normalize=True) * 100

print('Churn Counts:')
print(churn_counts.to_string())
print('\nChurn Percentages:')
print(churn_pct.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Bar chart ──
colors = ['#2ecc71', '#e74c3c']          # green = active, red = churned
bars = axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', width=0.5)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 30,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Customer Churn Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Number of Customers')

# ── Pie chart ──
wedge_props = {'edgecolor': 'white', 'linewidth': 2}
axes[1].pie(churn_counts.values, labels=churn_counts.index,
            autopct='%1.1f%%', colors=colors,
            startangle=140, wedgeprops=wedge_props,
            textprops={'fontsize': 12})
axes[1].set_title('Customer Churn Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Overall Churn Overview', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n📌 Observation: The dataset is nearly balanced — ~50.3% customers churned.')

### 5.2 👤 Age Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Overall age histogram ──
axes[0].hist(df['age'], bins=20, color='#3498db', edgecolor='white', alpha=0.85)
axes[0].axvline(df['age'].mean(), color='red', linestyle='--', linewidth=1.5,
                label=f'Mean = {df["age"].mean():.1f}')
axes[0].axvline(df['age'].median(), color='orange', linestyle='--', linewidth=1.5,
                label=f'Median = {df["age"].median():.1f}')
axes[0].set_title('Age Distribution (All Customers)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()

# ── Age group count ──
age_group_counts = df['age_group'].value_counts().sort_index()
axes[1].bar(age_group_counts.index.astype(str), age_group_counts.values,
            color='#9b59b6', edgecolor='white', alpha=0.85)
for i, v in enumerate(age_group_counts.values):
    axes[1].text(i, v + 15, str(v), ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('Customers by Age Group', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Count')

plt.suptitle('Age Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📌 Observation: Age is fairly uniformly distributed between 18 and 70, meaning Netflix has a wide customer age range.')

### 5.3 🚻 Gender Analysis

In [ ]:
gender_counts = df['gender'].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
palette = {'Female': '#e91e8c', 'Male': '#1e90ff', 'Other': '#f39c12'}
bars = ax.bar(gender_counts.index, gender_counts.values,
              color=[palette[g] for g in gender_counts.index],
              edgecolor='white', width=0.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 15,
            f'{int(bar.get_height()):,}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Customer Distribution by Gender', fontsize=13, fontweight='bold')
ax.set_xlabel('Gender')
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.show()

print('\n📌 Observation: Gender is roughly evenly split across Female, Male, and Other categories.')

### 5.4 📦 Subscription Type Analysis

In [ ]:
sub_counts = df['subscription_type'].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
colors_sub = ['#c0392b', '#16a085', '#2980b9']
bars = ax.bar(sub_counts.index, sub_counts.values,
              color=colors_sub, edgecolor='white', width=0.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 15,
            f'{int(bar.get_height()):,}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Customers by Subscription Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Subscription Type')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print('\n📌 Observation: All three plans — Basic ($8.99), Standard ($13.99), Premium ($17.99) — have roughly equal subscriber counts.')

### 5.5 🌍 Region Analysis

In [ ]:
region_counts = df['region'].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=region_counts.index, y=region_counts.values,
            palette='viridis', ax=ax, edgecolor='white')
for i, v in enumerate(region_counts.values):
    ax.text(i, v + 10, str(v), ha='center', fontsize=10, fontweight='bold')
ax.set_title('Customer Distribution by Region', fontsize=13, fontweight='bold')
ax.set_xlabel('Region')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print('\n📌 Observation: South America has the most customers, followed closely by Europe and North America.')

### 5.6 📱 Device Usage

In [ ]:
device_counts = df['device'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
sns.barplot(x=device_counts.index, y=device_counts.values,
            palette='coolwarm', ax=axes[0], edgecolor='white')
for i, v in enumerate(device_counts.values):
    axes[0].text(i, v + 8, str(v), ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Device Usage Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Device')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(device_counts.values, labels=device_counts.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('coolwarm', len(device_counts)),
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Device Share', fontsize=13, fontweight='bold')

plt.suptitle('Device Usage Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📌 Observation: Device usage is fairly balanced — Tablet is slightly most popular, followed by Laptop and Mobile.')

### 5.7 💳 Payment Method

In [ ]:
pay_counts = df['payment_method'].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=pay_counts.index, y=pay_counts.values,
            palette='Set2', ax=ax, edgecolor='white')
for i, v in enumerate(pay_counts.values):
    ax.text(i, v + 8, str(v), ha='center', fontsize=10, fontweight='bold')
ax.set_title('Payment Method Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Payment Method')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print('\n📌 Observation: All five payment methods are used almost equally, showing no dominant payment preference.')

### 5.8 🎬 Favourite Genre

In [ ]:
genre_counts = df['favorite_genre'].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
# Horizontal bar for readability
sns.barplot(x=genre_counts.values, y=genre_counts.index,
            palette='Spectral', ax=ax, edgecolor='white')
for i, v in enumerate(genre_counts.values):
    ax.text(v + 5, i, str(v), va='center', fontsize=10, fontweight='bold')
ax.set_title('Most Popular Genres', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Customers')
ax.set_ylabel('Genre')
plt.tight_layout()
plt.show()

print('\n📌 Observation: Drama leads as the most popular genre, followed closely by Documentary and Romance. Comedy has the fewest fans.')

---
<a id='6'></a>
## 6. 🔬 Feature Analysis vs Churn

Now we dig into each important feature and check: **do churned customers behave differently from active ones?**

### 6.1 🎥 Watch Hours vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
sns.boxplot(data=df, x='churn_label', y='watch_hours',
            palette={'Active': '#2ecc71', 'Churned': '#e74c3c'},
            ax=axes[0], width=0.4)
axes[0].set_title('Watch Hours by Churn Status', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Total Watch Hours')

# KDE (density) plot
for label, color in [('Active', '#2ecc71'), ('Churned', '#e74c3c')]:
    subset = df[df['churn_label'] == label]['watch_hours']
    sns.kdeplot(subset, ax=axes[1], color=color, fill=True, alpha=0.4, label=label)
axes[1].set_title('Watch Hours Distribution by Churn', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Watch Hours')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

# Show mean values
print('Mean Watch Hours by Churn Status:')
print(df.groupby('churn_label')['watch_hours'].mean().round(2))
print('\n📌 Observation: Churned and active customers have similar total watch hours, suggesting watch hours alone do not predict churn.')

### 6.2 🕐 Last Login Days vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot — shows distribution shape more clearly than box plot
sns.violinplot(data=df, x='churn_label', y='last_login_days',
               palette={'Active': '#2ecc71', 'Churned': '#e74c3c'},
               inner='quartile', ax=axes[0])
axes[0].set_title('Last Login Days — Violin Plot', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Days Since Last Login')

# KDE plot
for label, color in [('Active', '#2ecc71'), ('Churned', '#e74c3c')]:
    subset = df[df['churn_label'] == label]['last_login_days']
    sns.kdeplot(subset, ax=axes[1], color=color, fill=True, alpha=0.4, label=label)
axes[1].set_title('Last Login Days — Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Days Since Last Login')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Mean Last Login Days by Churn Status:')
print(df.groupby('churn_label')['last_login_days'].mean().round(2))
print('\n📌 Observation: Churned customers have a slightly higher average of days since last login — inactivity is a churn signal.')

### 6.3 💰 Monthly Fee vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Since monthly_fee has only 3 distinct values (plan prices), use countplot
sns.countplot(data=df, x='monthly_fee', hue='churn_label',
              palette={'Active': '#2ecc71', 'Churned': '#e74c3c'},
              ax=ax, edgecolor='white')
ax.set_title('Monthly Fee vs Churn Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Monthly Fee (USD)')
ax.set_ylabel('Count')
ax.set_xticklabels(['$8.99 (Basic)', '$13.99 (Standard)', '$17.99 (Premium)'])
ax.legend(title='Status')
plt.tight_layout()
plt.show()

# Churn rate by subscription
churn_by_sub = df.groupby('subscription_type')['churned'].mean() * 100
print('Churn Rate (%) by Subscription Type:')
print(churn_by_sub.round(2))
print('\n📌 Observation: Churn occurs across all price points. Check Section 8 for exact churn rates by subscription.')

### 6.4 👨‍👩‍👧 Number of Profiles vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.countplot(data=df, x='number_of_profiles', hue='churn_label',
              palette={'Active': '#2ecc71', 'Churned': '#e74c3c'},
              ax=ax, edgecolor='white')
ax.set_title('Number of Profiles vs Churn Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Profiles')
ax.set_ylabel('Count')
ax.legend(title='Status')
plt.tight_layout()
plt.show()

# Churn rate by number of profiles
churn_by_profiles = df.groupby('number_of_profiles')['churned'].mean() * 100
print('Churn Rate (%) by Number of Profiles:')
print(churn_by_profiles.round(2))
print('\n📌 Observation: Churn is spread across all profile counts. Accounts with fewer profiles don\'t show significantly higher churn.')

### 6.5 ⏱️ Avg Watch Time per Day vs Churn

In [ ]:
# Remove extreme outliers (top 1%) for cleaner visualisation
q99 = df['avg_watch_time_per_day'].quantile(0.99)
df_trimmed = df[df['avg_watch_time_per_day'] <= q99]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
sns.boxplot(data=df_trimmed, x='churn_label', y='avg_watch_time_per_day',
            palette={'Active': '#2ecc71', 'Churned': '#e74c3c'},
            ax=axes[0], width=0.4)
axes[0].set_title('Avg Watch Time/Day vs Churn (outliers removed)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Avg Hours / Day')

# KDE
for label, color in [('Active', '#2ecc71'), ('Churned', '#e74c3c')]:
    subset = df_trimmed[df_trimmed['churn_label'] == label]['avg_watch_time_per_day']
    sns.kdeplot(subset, ax=axes[1], color=color, fill=True, alpha=0.4, label=label)
axes[1].set_title('Avg Watch Time/Day — Distribution', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Hours / Day')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Mean Avg Watch Time per Day (hours):')
print(df.groupby('churn_label')['avg_watch_time_per_day'].mean().round(3))
print('\n📌 Observation: Active customers show slightly higher average daily watch time, indicating engagement level matters for retention.')

---
<a id='7'></a>
## 7. 🌡️ Correlation Heatmap

A correlation heatmap shows how strongly numeric features are related to each other.  
- Values close to **+1** = strong positive relationship  
- Values close to **−1** = strong negative relationship  
- Values close to **0** = weak or no relationship

In [ ]:
# Select only numeric columns for correlation
num_cols = ['age', 'watch_hours', 'last_login_days', 'monthly_fee',
            'number_of_profiles', 'avg_watch_time_per_day', 'churned']

corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # Hide upper triangle (redundant)

sns.heatmap(
    corr_matrix,
    annot=True,          # Show correlation numbers
    fmt='.2f',           # 2 decimal places
    cmap='RdYlGn',       # Red → Yellow → Green
    center=0,
    mask=mask,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Heatmap — Netflix Churn Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Focus on correlations with 'churned'
print('Correlation with Churned column (sorted):')
print(corr_matrix['churned'].drop('churned').sort_values(ascending=False).round(4))
print('\n📌 Observation: No single feature shows a very strong correlation with churn — multiple factors together drive it.')

---
<a id='8'></a>
## 8. 📈 Churn Rate by Category

Here we calculate the **churn rate** (%) for each category and visualise which segments are most at risk.

In [ ]:
# Helper function to plot churn rate for a given categorical column
def plot_churn_rate(column, title, ax, palette='viridis'):
    churn_rate = df.groupby(column)['churned'].mean() * 100
    churn_rate = churn_rate.sort_values(ascending=False)
    bars = ax.bar(churn_rate.index.astype(str), churn_rate.values,
                  color=sns.color_palette(palette, len(churn_rate)),
                  edgecolor='white')
    # Add a reference line at overall churn rate
    overall = df['churned'].mean() * 100
    ax.axhline(overall, color='red', linestyle='--', linewidth=1.3,
               label=f'Overall: {overall:.1f}%')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, max(churn_rate.values) + 10)
    ax.tick_params(axis='x', rotation=15)
    ax.legend(fontsize=9)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

plot_churn_rate('subscription_type', 'Churn Rate by Subscription Type', axes[0, 0], 'Set1')
plot_churn_rate('gender',            'Churn Rate by Gender',            axes[0, 1], 'Set2')
plot_churn_rate('region',            'Churn Rate by Region',            axes[0, 2], 'viridis')
plot_churn_rate('device',            'Churn Rate by Device',            axes[1, 0], 'coolwarm')
plot_churn_rate('payment_method',    'Churn Rate by Payment Method',    axes[1, 1], 'Spectral')
plot_churn_rate('favorite_genre',    'Churn Rate by Favourite Genre',   axes[1, 2], 'tab10')

plt.suptitle('Churn Rate (%) by Customer Segment', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate by age group
plot_churn_rate('age_group', 'Churn Rate by Age Group', axes[0], 'plasma')

# Churn rate by inactivity level
plot_churn_rate('inactivity_level', 'Churn Rate by Inactivity Level', axes[1], 'Reds')

plt.suptitle('Churn Rate by Age Group & Inactivity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📌 Key Observations:')
print('• Inactivity is a strong churn indicator — "Inactive (>30d)" segment has the highest churn rate.')
print('• Middle age groups (36-45, 46-55) tend to have slightly elevated churn rates.')

In [ ]:
# ── Stacked bar: Churn vs Active by Subscription Type ───────────────────────
sub_churn = df.groupby(['subscription_type', 'churn_label']).size().unstack(fill_value=0)
sub_churn_pct = sub_churn.div(sub_churn.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))
sub_churn_pct.plot(kind='bar', stacked=True, color=['#2ecc71', '#e74c3c'],
                   edgecolor='white', ax=ax, rot=0)
ax.set_title('Churn vs Active (%) by Subscription Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Subscription Type')
ax.set_ylabel('Percentage (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(title='Status', loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Grouped bar: Churn count by Region and Subscription Type ────────────────
pivot = df[df['churned'] == 1].groupby(['region', 'subscription_type']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(kind='bar', ax=ax, colormap='Set1', edgecolor='white', rot=15)
ax.set_title('Churned Customers by Region & Subscription Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Region')
ax.set_ylabel('Number of Churned Customers')
ax.legend(title='Subscription')
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: Watch Hours vs Avg Watch Time/Day, coloured by Churn ────────────
fig, ax = plt.subplots(figsize=(9, 6))

for label, color, marker in [('Active', '#2ecc71', 'o'), ('Churned', '#e74c3c', 'x')]:
    subset = df[df['churn_label'] == label]
    ax.scatter(subset['watch_hours'], subset['avg_watch_time_per_day'],
               alpha=0.3, s=18, color=color, label=label, marker=marker)

ax.set_title('Watch Hours vs Avg Daily Watch Time (by Churn)', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Watch Hours')
ax.set_ylabel('Avg Watch Time per Day (hours)')
ax.set_ylim(0, 5)
ax.legend(title='Status')
plt.tight_layout()
plt.show()

print('\n📌 Observation: Both churned and active customers are mixed across watch-hour ranges — no clear linear separation.')

---
<a id='9'></a>
## 9. 💡 Key Insights

Based on all the analysis above, here are the most important findings:

In [ ]:
# ── Summary Statistics Table ──────────────────────────────────────────────────
summary = df.groupby('churn_label').agg(
    Count                 = ('churned', 'count'),
    Avg_Age               = ('age', 'mean'),
    Avg_Watch_Hours       = ('watch_hours', 'mean'),
    Avg_Last_Login_Days   = ('last_login_days', 'mean'),
    Avg_Monthly_Fee       = ('monthly_fee', 'mean'),
    Avg_Profiles          = ('number_of_profiles', 'mean'),
    Avg_Daily_Watch_Hrs   = ('avg_watch_time_per_day', 'mean')
).round(2)

print('=== Summary Statistics: Active vs Churned Customers ===')
summary

In [ ]:
# ── Highest-churn segments ────────────────────────────────────────────────────
print('TOP CHURN INSIGHTS\n' + '='*50)

for col in ['subscription_type', 'gender', 'region', 'device',
            'payment_method', 'favorite_genre', 'inactivity_level']:
    top = df.groupby(col)['churned'].mean().idxmax()
    rate = df.groupby(col)['churned'].mean().max() * 100
    print(f'  Highest churn in {col:20s}: {top} ({rate:.1f}%)')

### 📌 Summary of Key Insights

| # | Insight |
|---|---|
| 1 | **~50.3% of customers churned** — the dataset is nearly balanced, meaning churn is a significant business problem. |
| 2 | **Inactivity is the strongest churn signal** — inactive customers (>30 days since login) churn at **75.1%**, compared to ~31% for recently active ones. |
| 3 | **Basic subscribers churn at ~61.8%** — the highest of the three plans, suggesting price-sensitive customers on the lowest tier are most likely to leave. |
| 4 | **Crypto payment users churn at ~59.7%** — the highest of all payment methods, possibly signalling less committed or more anonymous customers. |
| 5 | **Gender has minimal impact** — churn rates are close across Female, Male, and Other. |
| 6 | **Region differences are small** — no single region stands out dramatically. South America has the most customers overall. |
| 7 | **Watch hours and avg daily time** — active customers show slightly higher daily engagement, but the difference is modest. |
| 8 | **Device type doesn't strongly predict churn** — all devices are used by both churned and active customers equally. |
| 9 | **Drama is the favourite genre** — but genre preference doesn't strongly differentiate churners from active users. |
| 10 | **Correlations are weak** — no single feature is a silver bullet; churn is a multi-factor problem requiring a combination of signals. |

---
<a id='10'></a>
## 10. ✅ Conclusion

In this notebook, **Alyana Bhanu Chandar** performed a complete EDA on the Netflix Customer Churn dataset (5,000 records, 14 features).

### 🔑 What We Did
| Step | Action |
|---|---|
| Data Loading | Loaded the CSV and previewed its structure |
| Data Cleaning | Removed the ID column, validated ranges, and engineered 3 new features |
| EDA | Visualised distributions for all 14 columns |
| Feature vs Churn | Compared churned vs active customers across all key variables |
| Correlation | Built a heatmap to find inter-feature relationships |
| Churn Rate | Computed churn rate by every categorical segment |

### 🚀 Recommended Next Steps

1. **Build a Machine Learning Model** — train a Logistic Regression, Random Forest, or XGBoost classifier to predict churn.
2. **Feature Importance** — identify which features contribute most to the churn prediction model.
3. **Retention Strategies** — target the "Inactive (>30d)" segment with re-engagement campaigns (e.g., personalised email, discounts).
4. **A/B Testing** — experiment with different pricing tiers or content recommendations for high-risk segments.
5. **Time-Series Analysis** — if timestamp data becomes available, track churn trends over time.

---

> *This notebook was created by **Alyana Bhanu Chandar** as a beginner-friendly walkthrough of the Netflix Customer Churn dataset using Python data analysis best practices.*